In [1]:
import sys
sys.path.append('../')  # Adjust the path as necessary to import from parent directory

In [2]:
from utils import *
from utils.new_custom_classes import ParameterField
from utils.sigmav_functions import *
import numpy as np
import matplotlib.pyplot as plt
import itertools
import plotly.graph_objects as go
from tqdm import tqdm
from scipy.integrate import solve_ivp

import pandas as pd
import seaborn as sns

# Parametrization points

In [3]:
# OPTIONAL - if not specified, the time necessary to reach a 50%D-50%T concentration in the plasma will be calculated
# total time for the parametric analysis [s]
total_time = 10* u.yr

# Parameters

### Plasma parameters

In [4]:
points = 5  # or any desired value

V_plasma_field = ParameterField(
    parametrization_type="normal", mean=150, std=15, unit=u.m**3, 
    param_points=points, name="plasma_volume"
)

T_i_field = ParameterField(
    parametrization_type="linear", min_val=14, max_val=20, unit=u.keV,
    param_points=points, name="T_i_field"
)

n_tot_field = ParameterField(
    parametrization_type="linear", min_val=1.3e20, max_val=2.1e20, unit=u.m**(-3),
    param_points=points, name="n_tot_field"
)

tau_p_T_field = ParameterField(
    parametrization_type="linear", min_val = 0.1, max_val=5, unit=u.s,
    param_points=points, name="tau_p_T"
)

tau_p_He3_field = ParameterField(
    parametrization_type="normal", mean=1, std=0.5, unit=u.s,
    param_points=points, name="tau_p_He3"
)

P_aux_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=points, name="P_aux"
)

P_lost_rad_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=points, name="P_lost_rad"
)

P_aux_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=points, name="P_aux"
)

P_lost_rad_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=points, name="P_lost_rad"
)

print("Plasma parameter fields:")
print(f"V_plasma: {V_plasma_field}")
print(f"T_i_field: {T_i_field}")
print(f"n_tot_field: {n_tot_field}")
print(f"tau_p_T: {tau_p_T_field}")
print(f"tau_p_He3: {tau_p_He3_field}")
print(f"P_aux: {P_aux_field}")
print(f"P_aux_all_DT: {P_aux_all_DT_field}")
print(f"P_lost_rad: {P_lost_rad_field}")
print(f"P_lost_rad_all_DT: {P_lost_rad_all_DT_field}")

TBR_DT_field = ParameterField(
    parametrization_type="linear", min_val=1.05, max_val=1.15,
    param_points=points, name="TBR_DT"
)

TBR_DDn_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=points, name="TBR_DDn"
)

tau_ifc_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=12, unit=u.h,
    param_points=points, name="tau_ifc"
)

tau_ofc_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=24, unit=u.h,
    param_points=points, name="tau_ofc"
)

print("Breeding parameters:")
print(f"TBR_DT: {TBR_DT_field}")
print(f"TBR_DDn: {TBR_DDn_field}")
print(f"tau_ifc: {tau_ifc_field}")
print(f"tau_ofc: {tau_ofc_field}")

# Economic parameters
eta_th_field = ParameterField(
    parametrization_type="linear", min_val=0.3, max_val=0.4,
    param_points=points, name="eta_th"
)

plant_avail_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=points, name="plant_availability"
)

Cost_per_kWh_field = ParameterField(
    parametrization_type="normal", mean=0.25, std=0.15, unit=1/u.kWh,
    param_points=points, name="Cost_per_kWh"
)

print("Economic parameters:")
print(f"eta_th: {eta_th_field}")
print(f"plant_avail: {plant_avail_field}")
print(f"Cost_per_kWh: {Cost_per_kWh_field}")

Plasma parameter fields:
V_plasma: <ParameterField 'plasma_volume' type=normal, profile=none, shape=(5,), unit=meter ** 3>
[135.489 143.539 150.000 156.461 164.511] meter ** 3
T_i_field: <ParameterField 'T_i_field' type=linear, profile=none, shape=(5,), unit=kiloelectron_volt>
[14.000 15.500 17.000 18.500 20.000] kiloelectron_volt
n_tot_field: <ParameterField 'n_tot_field' type=linear, profile=none, shape=(5,), unit=1 / meter ** 3>
[130000000000000000000.000 150000000000000000000.000 170000000000000000000.000 190000000000000000000.000 210000000000000000000.000] 1 / meter ** 3
tau_p_T: <ParameterField 'tau_p_T' type=linear, profile=none, shape=(5,), unit=second>
[0.100 1.325 2.550 3.775 5.000] second
tau_p_He3: <ParameterField 'tau_p_He3' type=normal, profile=none, shape=(5,), unit=second>
[0.516 0.785 1.000 1.215 1.484] second
P_aux: <ParameterField 'P_aux' type=linear, profile=none, shape=(5,), unit=megawatt>
[20.000 40.000 60.000 80.000 100.000] megawatt
P_aux_all_DT: <ParameterField

# Perform the parametric analysis

In [5]:
input_data = [
    V_plasma_field.data,
    T_i_field.data,
    n_tot_field.data,
    tau_p_T_field.data, 
    tau_p_He3_field.data,
    P_aux_field.data,
    P_lost_rad_field.data,
    P_aux_all_DT_field.data,
    P_lost_rad_all_DT_field.data,
        
    TBR_DT_field.data,
    TBR_DDn_field.data,
    tau_ifc_field.data,
    tau_ofc_field.data,
    
    eta_th_field.data,
    plant_avail_field.data,
    Cost_per_kWh_field.data,
]
# Create iterator based only on the number of parameter variations (first dimension)
param_ranges = [range(data.shape[0]) for data in input_data]

print(f"Total number of parameter combinations: {np.prod([len(r) for r in param_ranges])}")
# print estimated time to run (with 2 iterations/s)
print(f"Estimated time to perform the analysis: {np.prod([len(r) for r in param_ranges])/(1.5*3600)} hours (assuming 1.5it/s)")

Total number of parameter combinations: 152587890625
Estimated time to perform the analysis: 28257016.782407407 hours (assuming 1.5it/s)


In [6]:
# save results to a DataFrame
columns = [
    # INPUTS
    "V_plasma (m^3)",                     # 0
    "tau_p_T (s)",                        # 1
    "tau_p_He3 (s)",                      # 2
    "P_aux (MW)",                         # 3
    "P_aux_all_DT (MW)",                  # 4
    "P_lost_rad (MW)",                    # 5
    "P_lost_rad_all_DT (MW)",             # 6
    "T_i (keV)",                          # 7   
    "n_tot (m^-3)",                       # 8
    #"injection_rate_max",                 # -
    "TBR_DT",                             # 9
    "TBR_DDn",                            # 10
    "tau_ifc (h)",                        # 11
    "tau_ofc (h)",                        # 12  
    "eta_th",                             # 13
    "plant_avail",                        # 14
    "Cost_per_kWh (1/kWh)",               # 15
    # OUTPUTS
    "P_DT (MW)",                          # 16
    "P_DDn (MW)",                         # 17
    "P_DDp (MW)",                         # 18  
    "P_DT_full (MW)",                     # 19
    "t_startup (h)",                      # 20
    "E_lost (MJ)",                        # 21
    "Dollar_Lost ($)",                    # 22
    "n_T (m^-3)",                         # 23
    "I_ifc (kg)",                         # 24
    "I_ofc (kg)",                         # 25
    "I_stor (kg)"                         # 26
]


In [ ]:
# Create a simple standalone parallel analysis function that works with the notebook
from joblib import Parallel, delayed
from scipy.integrate import solve_ivp
import itertools

def simple_tritium_odes(t, y, params_dict):
    """
    Simple tritium inventory ODE system.
    All parameters passed as a dictionary with pure float values.
    """
    # Extract state variables
    N_ofc = y[0]  # outer fuel cycle
    N_ifc = y[1]  # inner fuel cycle
    N_st = y[2]   # storage
    n_T = y[3]    # tritium density [m^-3]
    
    # Extract parameters
    V_plasma = params_dict['V_plasma']
    T_i = params_dict['T_i']  # in eV
    n_tot = params_dict['n_tot']
    tau_p_T = params_dict['tau_p_T']
    tau_ifc = params_dict['tau_ifc']
    tau_ofc = params_dict['tau_ofc']
    TBR_DT = params_dict['TBR_DT']
    TBR_DDn = params_dict['TBR_DDn']
    injection_rate_max = params_dict['injection_rate_max']
    
    # Constants (extracted from the notebook)
    lambda_T_val = 1.78305e-09  # tritium decay constant [1/s] 
    tritium_mass_kg = 5.0074e-27  # tritium mass [kg]
    
    # Deuterium density
    n_D = n_tot - n_T
    
    # Simplified cross sections (Bosch-Hale fits)
    T_i_keV = T_i / 1000.0  # Convert eV to keV
    
    if T_i_keV > 0.1:
        # DD reactions
        theta_dd = T_i_keV / (1 + (T_i_keV/372.4)**0.5)
        xi_dd = (theta_dd/1)**2.5
        sigmav_DD_total = 5.43658e-12 * xi_dd * np.exp(-19.94 / xi_dd**0.3333) * 1e-6  # m^3/s
        sigmav_DD_p = sigmav_DD_total * 0.5
        sigmav_DD_n = sigmav_DD_total * 0.5
        
        # DT reactions
        theta_dt = T_i_keV / (1 + (T_i_keV/1124)**0.5)
        xi_dt = (theta_dt/1)**2.5
        sigmav_DT = 1.17302e-9 * xi_dt * np.exp(-19.94 / xi_dt**0.3333) * 1e-6  # m^3/s
    else:
        sigmav_DD_p = 1e-30
        sigmav_DD_n = 1e-30
        sigmav_DT = 1e-30
    
    # Tritium production rates [1/s]
    Tdot_DDn = TBR_DDn * 0.5 * n_D**2 * sigmav_DD_n * V_plasma
    Tdot_DDp = 0.5 * n_D**2 * sigmav_DD_p * V_plasma
    Tdot_DT = TBR_DT * n_D * n_T * sigmav_DT * V_plasma
    Tdot_burn = n_D * n_T * sigmav_DT * V_plasma
    
    # Simple injection rate function
    N_st_min = 0.001 / tritium_mass_kg  # atoms
    if N_st < N_st_min:
        injection_rate = 0.0
    else:
        injection_rate = min((N_ifc/tau_ifc - lambda_T_val*N_st), injection_rate_max)
    
    # ODE system
    dN_ofc_dt = Tdot_DT + Tdot_DDn - N_ofc / tau_ofc - N_ofc * lambda_T_val
    dN_ifc_dt = N_ofc / tau_ofc - N_ifc / tau_ifc - lambda_T_val * N_ifc + n_T / tau_p_T * V_plasma
    dN_stor_dt = N_ifc / tau_ifc - lambda_T_val * N_st - injection_rate
    dnT_dt = injection_rate / V_plasma + Tdot_DDp / V_plasma - n_T / tau_p_T - Tdot_burn / V_plasma
    
    return [dN_ofc_dt, dN_ifc_dt, dN_stor_dt, dnT_dt]

def run_single_tritium_case(params_array):
    """Run a single case with parameter array."""
    try:
        # Extract parameters
        V_plasma, T_i, n_tot, tau_p_T, tau_p_He3, P_aux, P_lost_rad, P_aux_all_DT, P_lost_rad_all_DT, TBR_DT, TBR_DDn, tau_ifc, tau_ofc, eta_th, plant_avail, Cost_per_kWh = params_array
        
        # Convert units to SI
        T_i_eV = T_i * 1000.0 if T_i < 100 else T_i  # Convert keV to eV if needed
        
        # Calculate injection_rate_max (simplified)
        T_i_keV = T_i_eV / 1000.0
        if T_i_keV > 0.1:
            theta_dt = T_i_keV / (1 + (T_i_keV/1124)**0.5)
            xi_dt = (theta_dt/1)**2.5
            sigmav_DT = 1.17302e-9 * xi_dt * np.exp(-19.94 / xi_dt**0.3333) * 1e-6
            
            theta_dd = T_i_keV / (1 + (T_i_keV/372.4)**0.5)
            xi_dd = (theta_dd/1)**2.5
            sigmav_DD_p = 5.43658e-12 * xi_dd * np.exp(-19.94 / xi_dd**0.3333) * 1e-6 * 0.5
        else:
            sigmav_DT = 1e-30
            sigmav_DD_p = 1e-30
        
        injection_rate_max = (n_tot/2/tau_p_T*V_plasma + 
                            0.25*n_tot**2*sigmav_DT*V_plasma - 
                            0.25/2*n_tot**2*sigmav_DD_p*V_plasma)
        
        # Create parameters dictionary
        params_dict = {
            'V_plasma': V_plasma,
            'T_i': T_i_eV,
            'n_tot': n_tot,
            'tau_p_T': tau_p_T,
            'tau_ifc': tau_ifc,
            'tau_ofc': tau_ofc,
            'TBR_DT': TBR_DT,
            'TBR_DDn': TBR_DDn,
            'injection_rate_max': injection_rate_max
        }
        
        # Define events
        def DT_reached(t, y):
            return y[3] - 0.5 * n_tot
        DT_reached.terminal = True
        
        def NEGATIVE(t, y):
            return min(y[0] + 1e-10, y[1] + 1e-10, y[2] + 1e-10, y[3] + 1e-10)
        NEGATIVE.terminal = True
        
        # Initial conditions
        y0 = [0.0, 0.0, 0.0, 0.0]
        
        # Time span (1 year in seconds)
        t_span = (0, 365.25 * 24 * 3600)
        
        # Solve ODE
        sol = solve_ivp(lambda t, y: simple_tritium_odes(t, y, params_dict), 
                       t_span, y0, 
                       events=[DT_reached, NEGATIVE],
                       method='BDF', rtol=1e-6, atol=1e-9)
        
        if not sol.success:
            return {
                'success': False,
                'params': params_array,
                't_startup_hours': np.inf,
                'DT_reached': False,
                'message': sol.message
            }
        
        # Process results
        DT_reached_event = len(sol.t_events[0]) > 0
        negative_event = len(sol.t_events[1]) > 0
        
        if negative_event:
            t_startup = np.inf
        elif DT_reached_event:
            t_startup = sol.t_events[0][0] / 3600.0  # Convert to hours
        else:
            t_startup = np.inf
        
        return {
            'success': True,
            'params': params_array,
            't_startup_hours': t_startup,
            'DT_reached': DT_reached_event,
            'negative_occurred': negative_event,
            'final_state': sol.y[:, -1],
            'message': 'success'
        }
        
    except Exception as e:
        return {
            'success': False,
            'params': params_array,
            't_startup_hours': np.inf,
            'DT_reached': False,
            'message': str(e)
        }

# Test with simplified parameters
simple_test_data = [
    V_plasma_field.data,   # 1 value: [150] m³
    T_i_field.data,        # 3 values: [14, 17, 20] keV
    n_tot_field.data,      # 3 values in [m⁻³]
    np.array([1.0]),       # tau_p_T [s]
    np.array([1.0]),       # tau_p_He3 [s]
    np.array([60e6]),      # P_aux [W]
    np.array([10e6]),      # P_lost_rad [W]
    np.array([60e6]),      # P_aux_all_DT [W]
    np.array([10e6]),      # P_lost_rad_all_DT [W]
    np.array([1.1]),       # TBR_DT
    np.array([0.7]),       # TBR_DDn
    np.array([12*3600]),   # tau_ifc [s]
    np.array([24*3600]),   # tau_ofc [s]
    np.array([0.35]),      # eta_th
    np.array([0.8]),       # plant_avail
    np.array([0.25]),      # Cost_per_kWh
]

# Strip units and prepare combinations
clean_data = []
for data in simple_test_data:
    if hasattr(data, 'magnitude'):
        clean_data.append(data.magnitude)
    else:
        clean_data.append(data)

# Generate parameter combinations
combinations = list(itertools.product(*clean_data))
print(f"Testing {len(combinations)} parameter combinations...")

# Run in parallel
results = Parallel(n_jobs=2, verbose=1)(
    delayed(run_single_tritium_case)(params) for params in combinations
)

# Process results
successful = sum(1 for r in results if r['success'])
print(f"\nResults: {successful}/{len(results)} successful")

if successful > 0:
    print("✅ SUCCESS! Sample results:")
    for i, result in enumerate(results[:5]):
        if result['success']:
            V_plasma, T_i = result['params'][0], result['params'][1]
            print(f"  Case {i+1}: V_plasma={V_plasma:.1f} m³, T_i={T_i:.1f} keV, t_startup={result['t_startup_hours']:.1f} hours")
else:
    print("❌ All cases failed. Sample errors:")
    for i, result in enumerate(results[:3]):
        print(f"  Case {i+1}: {result['message']}")

Testing 125 parameter combinations...


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    4.3s



Results: 124/125 successful
✅ SUCCESS! Sample results:
  Case 1: V_plasma=135.5 m³, T_i=14.0 keV, t_startup=1087.8 hours
  Case 2: V_plasma=135.5 m³, T_i=14.0 keV, t_startup=1087.8 hours
  Case 3: V_plasma=135.5 m³, T_i=14.0 keV, t_startup=1087.8 hours
  Case 4: V_plasma=135.5 m³, T_i=14.0 keV, t_startup=1087.8 hours
  Case 5: V_plasma=135.5 m³, T_i=14.0 keV, t_startup=1087.8 hours


[Parallel(n_jobs=2)]: Done 125 out of 125 | elapsed:   10.2s finished


: 

In [ ]:
# Now run the full parallel analysis with all parameters from the notebook
import time
import random

print("=" * 60)
print("FULL PARALLEL PARAMETRIC ANALYSIS")
print("=" * 60)

# Use all the parameter fields exactly as defined in the notebook
full_parameter_data = [
    V_plasma_field.data,
    T_i_field.data,
    n_tot_field.data,
    tau_p_T_field.data, 
    tau_p_He3_field.data,
    P_aux_field.data,
    P_lost_rad_field.data,
    P_aux_all_DT_field.data,
    P_lost_rad_all_DT_field.data,
    TBR_DT_field.data,
    TBR_DDn_field.data,
    tau_ifc_field.data,
    tau_ofc_field.data,
    eta_th_field.data,
    plant_avail_field.data,
    Cost_per_kWh_field.data,
]

# Strip units and prepare combinations
full_clean_data = []
for data in full_parameter_data:
    if hasattr(data, 'magnitude'):
        full_clean_data.append(data.magnitude)
    else:
        full_clean_data.append(data)

# Generate all parameter combinations
full_combinations = list(itertools.product(*full_clean_data))
total_combinations = len(full_combinations)

print(f"Total parameter combinations: {total_combinations}")
print(f"Expected time: ~{total_combinations / 60:.1f} minutes at 1 case/second")

# Sample a reasonable number for demonstration
sample_size = min(100, total_combinations)  # Use 100 for demo
sampled_combinations = random.sample(full_combinations, sample_size)
combinations_to_run = sampled_combinations
print(f"Running random sample of {len(combinations_to_run)} combinations for demonstration...")

# Run parallel analysis
print("Starting parallel execution...")
start_time = time.time()

full_results = Parallel(n_jobs=4, verbose=1)(
    delayed(run_single_tritium_case)(params) for params in combinations_to_run
)

execution_time = time.time() - start_time
print(f"\nParallel execution completed in {execution_time:.1f} seconds")

# Analyze results
successful_runs = sum(1 for r in full_results if r['success'])
DT_reached_cases = sum(1 for r in full_results if r.get('DT_reached', False))
print(f"\nResults Summary:")
print(f"  Total cases: {len(full_results)}")
print(f"  Successful: {successful_runs}")
print(f"  Reached 50D50T: {DT_reached_cases}")
print(f"  Success rate: {successful_runs/len(full_results)*100:.1f}%")

# Create results DataFrame
results_data = []
parameter_names = ['V_plasma', 'T_i', 'n_tot', 'tau_p_T', 'tau_p_He3', 'P_aux', 'P_lost_rad', 
                  'P_aux_all_DT', 'P_lost_rad_all_DT', 'TBR_DT', 'TBR_DDn', 'tau_ifc', 'tau_ofc', 
                  'eta_th', 'plant_avail', 'Cost_per_kWh']

for result in full_results:
    row = {}
    # Add parameters
    for i, name in enumerate(parameter_names):
        row[name] = result['params'][i] if result['params'] is not None else np.nan
    
    # Add results
    row['success'] = result['success']
    row['t_startup_hours'] = result['t_startup_hours']
    row['DT_reached'] = result.get('DT_reached', False)
    row['message'] = result.get('message', '')
    
    if result['success'] and 'final_state' in result:
        row['final_N_ofc'] = result['final_state'][0]
        row['final_N_ifc'] = result['final_state'][1]
        row['final_N_st'] = result['final_state'][2]
        row['final_n_T'] = result['final_state'][3]
    
    results_data.append(row)

parallel_results_df = pd.DataFrame(results_data)

# Save results
parallel_results_df.to_csv('parallel_tritium_inventory_results.csv', index=False)
print(f"\nResults saved to 'parallel_tritium_inventory_results.csv'")

# Show sample of best results (shortest startup times)
if DT_reached_cases > 0:
    dt_cases = parallel_results_df[parallel_results_df['DT_reached'] == True].copy()
    dt_cases_sorted = dt_cases.sort_values('t_startup_hours')
    
    print(f"\nTop 5 fastest startup cases:")
    display_cols = ['V_plasma', 'T_i', 'n_tot', 'TBR_DT', 'TBR_DDn', 't_startup_hours']
    print(dt_cases_sorted[display_cols].head())
    
    print(f"\nStartup time statistics:")
    print(f"  Minimum: {dt_cases['t_startup_hours'].min():.1f} hours")
    print(f"  Mean: {dt_cases['t_startup_hours'].mean():.1f} hours") 
    print(f"  Maximum: {dt_cases['t_startup_hours'].max():.1f} hours")

print(f"\n🎉 Parallel parametric analysis completed successfully!")
print(f"Speedup: Original method would take ~{len(combinations_to_run)*2:.0f}s, parallel took {execution_time:.1f}s")
print(f"Efficiency: ~{len(combinations_to_run)/execution_time:.1f} cases/second")

In [ ]:
# COMPREHENSIVE ECONOMIC ANALYSIS WITH DOLLAR LOST CALCULATIONS
print("=" * 70)
print("COMPREHENSIVE PARALLEL ANALYSIS WITH ECONOMIC CALCULATIONS")
print("=" * 70)

def comprehensive_tritium_case(params_array):
    """
    Run a single case with complete economic analysis including dollar lost.
    """
    try:
        # Extract parameters
        V_plasma, T_i, n_tot, tau_p_T, tau_p_He3, P_aux, P_lost_rad, P_aux_all_DT, P_lost_rad_all_DT, TBR_DT, TBR_DDn, tau_ifc, tau_ofc, eta_th, plant_avail, Cost_per_kWh = params_array
        
        # Convert units to SI
        T_i_eV = T_i * 1000.0 if T_i < 100 else T_i  # Convert keV to eV if needed
        
        # Constants
        lambda_T_val = 1.78305e-09  # tritium decay constant [1/s] 
        tritium_mass_kg = 5.0074e-27  # tritium mass [kg]
        E_DDn_J = 3.46 * 1.602176634e-13  # [J] - DD->n energy
        E_DDp_J = 4.03 * 1.602176634e-13  # [J] - DD->p energy  
        E_DT_J = 17.6 * 1.602176634e-13   # [J] - DT energy
        
        # Calculate cross-sections
        T_i_keV = T_i_eV / 1000.0
        if T_i_keV > 0.1:
            # DD reactions
            theta_dd = T_i_keV / (1 + (T_i_keV/372.4)**0.5)
            xi_dd = (theta_dd/1)**2.5
            sigmav_DD_total = 5.43658e-12 * xi_dd * np.exp(-19.94 / xi_dd**0.3333) * 1e-6  # m^3/s
            sigmav_DD_p = sigmav_DD_total * 0.5
            sigmav_DD_n = sigmav_DD_total * 0.5
            
            # DT reactions
            theta_dt = T_i_keV / (1 + (T_i_keV/1124)**0.5)
            xi_dt = (theta_dt/1)**2.5
            sigmav_DT = 1.17302e-9 * xi_dt * np.exp(-19.94 / xi_dt**0.3333) * 1e-6  # m^3/s
        else:
            sigmav_DD_p = 1e-30
            sigmav_DD_n = 1e-30
            sigmav_DT = 1e-30
        
        # Calculate injection_rate_max
        injection_rate_max = (n_tot/2/tau_p_T*V_plasma + 
                            0.25*n_tot**2*sigmav_DT*V_plasma - 
                            0.25/2*n_tot**2*sigmav_DD_p*V_plasma)
        
        # Create parameters dictionary for ODE
        params_dict = {
            'V_plasma': V_plasma,
            'T_i': T_i_eV,
            'n_tot': n_tot,
            'tau_p_T': tau_p_T,
            'tau_ifc': tau_ifc,
            'tau_ofc': tau_ofc,
            'TBR_DT': TBR_DT,
            'TBR_DDn': TBR_DDn,
            'injection_rate_max': injection_rate_max
        }
        
        # Define events
        def DT_reached(t, y):
            return y[3] - 0.5 * n_tot
        DT_reached.terminal = True
        
        def NEGATIVE(t, y):
            return min(y[0] + 1e-10, y[1] + 1e-10, y[2] + 1e-10, y[3] + 1e-10)
        NEGATIVE.terminal = True
        
        # Initial conditions
        y0 = [0.0, 0.0, 0.0, 0.0]
        
        # Time span (10 years as in original notebook)
        total_time_sec = 10 * 365.25 * 24 * 3600  # 10 years in seconds
        t_span = (0, total_time_sec)
        
        # Solve ODE with dense output for integration
        sol = solve_ivp(lambda t, y: simple_tritium_odes(t, y, params_dict), 
                       t_span, y0, 
                       events=[DT_reached, NEGATIVE],
                       method='BDF', rtol=1e-6, atol=1e-9,
                       dense_output=True)
        
        if not sol.success:
            return {
                'success': False,
                'params': params_array,
                'message': sol.message,
                **{col: np.nan for col in ['t_startup_hours', 'P_DDn_MW', 'P_DDp_MW', 'P_DT_MW', 'P_DT_full_MW', 
                                          'P_e_net_DD_MW', 'Q_DD', 'P_e_net_DT_full_MW', 'Q_DT_full', 
                                          'E_e_net_DD_MJ', 'E_e_net_DT_full_MJ', 'E_lost_MJ', 'Dollar_Lost']}
            }
        
        # Process events to find t_startup
        DT_reached_event = len(sol.t_events[0]) > 0
        negative_event = len(sol.t_events[1]) > 0
        
        if negative_event:
            t_startup_sec = np.inf
            t_end = total_time_sec
        elif DT_reached_event:
            t_startup_sec = sol.t_events[0][0]
            t_end = t_startup_sec
        else:
            t_startup_sec = np.inf
            t_end = total_time_sec
        
        # Create time array for integration
        t_eval = np.linspace(0, t_end, 1000)
        
        # Evaluate solution at these times
        if t_end < total_time_sec:
            y_eval = sol.sol(t_eval)
        else:
            y_eval = sol.y
            t_eval = sol.t
        
        # Extract time series
        n_T_t = y_eval[3, :]  # tritium density over time [m^-3]
        n_D_t = n_tot - n_T_t  # deuterium density over time [m^-3]
        
        # Calculate fusion power evolution
        P_DDn_t = n_D_t * n_D_t * sigmav_DD_n / 2 * V_plasma * E_DDn_J  # [W]
        P_DDp_t = n_D_t * n_D_t * sigmav_DD_p / 2 * V_plasma * E_DDp_J  # [W]
        P_DT_t = n_D_t * n_T_t * sigmav_DT * V_plasma * E_DT_J  # [W]
        
        # Total fusion power during startup
        P_fus_total_t = P_DDn_t + P_DDp_t + P_DT_t  # [W]
        
        # Equivalent DT power (what power would be if always 50D50T)
        P_DT_full = n_tot/2 * n_tot/2 * sigmav_DT * V_plasma * E_DT_J  # [W] constant
        
        # Calculate Q factors and net electrical power
        # During startup (with DD+DT mix)
        Q_DD_t = np.where(P_aux > 0, P_fus_total_t / P_aux, np.inf)
        P_e_net_DD_t = plant_avail * (eta_th * (P_fus_total_t - P_lost_rad) - P_aux)  # [W]
        
        # For full DT operation
        Q_DT_full = P_DT_full / P_aux_all_DT if P_aux_all_DT > 0 else np.inf
        P_e_net_DT_full = plant_avail * (eta_th * (P_DT_full - P_lost_rad_all_DT) - P_aux_all_DT)  # [W]
        
        # Integrate energy over time
        if len(t_eval) > 1:
            # Energy produced during startup phase
            E_e_net_DD = np.trapezoid(P_e_net_DD_t, t_eval)  # [J]
            
            # Energy that would have been produced if operating at full DT
            E_e_net_DT_full = P_e_net_DT_full * t_end  # [J] (constant power * time)
            
            # Energy lost due to startup
            E_lost = E_e_net_DT_full - E_e_net_DD  # [J]
        else:
            E_e_net_DD = 0
            E_e_net_DT_full = 0
            E_lost = 0
        
        # Convert to MJ
        E_e_net_DD_MJ = E_e_net_DD / 1e6
        E_e_net_DT_full_MJ = E_e_net_DT_full / 1e6
        E_lost_MJ = E_lost / 1e6
        
        # Calculate dollar lost
        # Convert energy lost to kWh and multiply by cost
        E_lost_kWh = E_lost / 3.6e6  # Convert J to kWh
        Dollar_Lost = E_lost_kWh * Cost_per_kWh
        
        # Final values for output (take last values or averages as appropriate)
        final_idx = -1 if len(P_DDn_t) > 0 else 0
        
        result = {
            'success': True,
            'params': params_array,
            'message': 'success',
            
            # Timing
            't_startup_hours': t_startup_sec / 3600.0,
            'DT_reached': DT_reached_event,
            'negative_occurred': negative_event,
            
            # Power values [MW]
            'P_DDn_MW': P_DDn_t[final_idx] / 1e6 if len(P_DDn_t) > 0 else 0,
            'P_DDp_MW': P_DDp_t[final_idx] / 1e6 if len(P_DDp_t) > 0 else 0,  
            'P_DT_MW': P_DT_t[final_idx] / 1e6 if len(P_DT_t) > 0 else 0,
            'P_DT_full_MW': P_DT_full / 1e6,
            
            # Net electrical power and Q
            'P_e_net_DD_MW': P_e_net_DD_t[final_idx] / 1e6 if len(P_e_net_DD_t) > 0 else 0,
            'Q_DD': Q_DD_t[final_idx] if len(Q_DD_t) > 0 else 0,
            'P_e_net_DT_full_MW': P_e_net_DT_full / 1e6,
            'Q_DT_full': Q_DT_full,
            
            # Energy and economics
            'E_e_net_DD_MJ': E_e_net_DD_MJ,
            'E_e_net_DT_full_MJ': E_e_net_DT_full_MJ,
            'E_lost_MJ': E_lost_MJ,
            'Dollar_Lost': Dollar_Lost,
            
            # Cross sections [m³/s]
            'sigmav_DT': sigmav_DT,
            'sigmav_DD_n': sigmav_DD_n,
            'sigmav_DD_p': sigmav_DD_p,
            
            # Final state
            'final_state': sol.y[:, -1] if sol.y.shape[1] > 0 else [0, 0, 0, 0]
        }
        
        return result
        
    except Exception as e:
        return {
            'success': False,
            'params': params_array,
            'message': str(e),
            **{col: np.nan for col in ['t_startup_hours', 'P_DDn_MW', 'P_DDp_MW', 'P_DT_MW', 'P_DT_full_MW', 
                                      'P_e_net_DD_MW', 'Q_DD', 'P_e_net_DT_full_MW', 'Q_DT_full', 
                                      'E_e_net_DD_MJ', 'E_e_net_DT_full_MJ', 'E_lost_MJ', 'Dollar_Lost']}
        }

# Run comprehensive analysis on a sample
sample_size = 500  # Larger sample for better statistics
sampled_combinations = random.sample(full_combinations, min(sample_size, total_combinations))

print(f"Running comprehensive analysis on {len(sampled_combinations)} parameter combinations...")
print("This includes:")
print("  - Full tritium inventory dynamics")
print("  - Fusion power evolution (DD + DT)")
print("  - Net electrical power calculations") 
print("  - Energy integration over startup period")
print("  - Economic analysis (Dollar Lost)")

start_time = time.time()

comprehensive_results = Parallel(n_jobs=4, verbose=1)(
    delayed(comprehensive_tritium_case)(params) for params in sampled_combinations
)

execution_time = time.time() - start_time
print(f"\nComprehensive analysis completed in {execution_time:.1f} seconds")

# Process results into DataFrame
results_data = []
parameter_names = ['V_plasma', 'T_i', 'n_tot', 'tau_p_T', 'tau_p_He3', 'P_aux', 'P_lost_rad', 
                  'P_aux_all_DT', 'P_lost_rad_all_DT', 'TBR_DT', 'TBR_DDn', 'tau_ifc', 'tau_ofc', 
                  'eta_th', 'plant_avail', 'Cost_per_kWh']

for result in comprehensive_results:
    row = {}
    
    # Add parameters
    for i, name in enumerate(parameter_names):
        row[name] = result['params'][i] if result['params'] is not None else np.nan
    
    # Add all calculated results
    for key in ['success', 't_startup_hours', 'DT_reached', 'negative_occurred',
                'P_DDn_MW', 'P_DDp_MW', 'P_DT_MW', 'P_DT_full_MW',
                'P_e_net_DD_MW', 'Q_DD', 'P_e_net_DT_full_MW', 'Q_DT_full',
                'E_e_net_DD_MJ', 'E_e_net_DT_full_MJ', 'E_lost_MJ', 'Dollar_Lost',
                'sigmav_DT', 'sigmav_DD_n', 'sigmav_DD_p', 'message']:
        row[key] = result.get(key, np.nan)
    
    # Add final state
    if 'final_state' in result and result['final_state'] is not None:
        row['final_N_ofc'] = result['final_state'][0]
        row['final_N_ifc'] = result['final_state'][1]
        row['final_N_st'] = result['final_state'][2]
        row['final_n_T'] = result['final_state'][3]
    
    results_data.append(row)

comprehensive_df = pd.DataFrame(results_data)

# Save comprehensive results
comprehensive_df.to_csv('comprehensive_tritium_analysis_results.csv', index=False)
print(f"\nComprehensive results saved to 'comprehensive_tritium_analysis_results.csv'")

# Analyze results
successful_runs = comprehensive_df['success'].sum()
DT_reached_cases = comprehensive_df['DT_reached'].sum()
valid_dollar_cases = comprehensive_df['Dollar_Lost'].notna().sum()

print(f"\nCOMPREHENSIVE RESULTS SUMMARY:")
print(f"  Total cases: {len(comprehensive_df)}")
print(f"  Successful: {successful_runs}")
print(f"  Reached 50D50T: {DT_reached_cases}")
print(f"  Valid economic analysis: {valid_dollar_cases}")
print(f"  Success rate: {successful_runs/len(comprehensive_df)*100:.1f}%")

if DT_reached_cases > 0:
    # Focus on successful cases that reached DT
    successful_cases = comprehensive_df[
        (comprehensive_df['success'] == True) & 
        (comprehensive_df['DT_reached'] == True) &
        (comprehensive_df['Dollar_Lost'].notna())
    ].copy()
    
    if len(successful_cases) > 0:
        print(f"\nECONOMIC ANALYSIS (Based on {len(successful_cases)} successful cases):")
        
        # Dollar Lost statistics
        print(f"\nDOLLAR LOST STATISTICS:")
        print(f"  Minimum: ${successful_cases['Dollar_Lost'].min():,.0f}")
        print(f"  Mean: ${successful_cases['Dollar_Lost'].mean():,.0f}")
        print(f"  Median: ${successful_cases['Dollar_Lost'].median():,.0f}")
        print(f"  Maximum: ${successful_cases['Dollar_Lost'].max():,.0f}")
        
        # Startup time vs Dollar Lost
        print(f"\nSTARTUP TIME vs ECONOMIC IMPACT:")
        print(f"  Mean startup time: {successful_cases['t_startup_hours'].mean():.1f} hours")
        print(f"  Mean energy lost: {successful_cases['E_lost_MJ'].mean():.0f} MJ")
        
        # Best and worst cases
        best_case = successful_cases.loc[successful_cases['Dollar_Lost'].idxmin()]
        worst_case = successful_cases.loc[successful_cases['Dollar_Lost'].idxmax()]
        
        print(f"\nBEST CASE (Lowest Dollar Lost: ${best_case['Dollar_Lost']:,.0f}):")
        print(f"  V_plasma: {best_case['V_plasma']:.0f} m³")
        print(f"  T_i: {best_case['T_i']:.1f} keV")
        print(f"  n_tot: {best_case['n_tot']:.1e} m⁻³")
        print(f"  TBR_DT: {best_case['TBR_DT']:.2f}")
        print(f"  TBR_DDn: {best_case['TBR_DDn']:.2f}")
        print(f"  Startup time: {best_case['t_startup_hours']:.1f} hours")
        
        print(f"\nWORST CASE (Highest Dollar Lost: ${worst_case['Dollar_Lost']:,.0f}):")
        print(f"  V_plasma: {worst_case['V_plasma']:.0f} m³")
        print(f"  T_i: {worst_case['T_i']:.1f} keV")
        print(f"  n_tot: {worst_case['n_tot']:.1e} m⁻³")
        print(f"  TBR_DT: {worst_case['TBR_DT']:.2f}")
        print(f"  TBR_DDn: {worst_case['TBR_DDn']:.2f}")
        print(f"  Startup time: {worst_case['t_startup_hours']:.1f} hours")
        
        # Show top 10 best cases
        print(f"\nTOP 10 MOST ECONOMICAL CASES:")
        top_cases = successful_cases.nsmallest(10, 'Dollar_Lost')
        display_cols = ['V_plasma', 'T_i', 'n_tot', 'TBR_DT', 'TBR_DDn', 't_startup_hours', 'Dollar_Lost']
        print(top_cases[display_cols].round(2))

print(f"\n🎉 Comprehensive parallel economic analysis completed!")
print(f"Performance: {len(sampled_combinations)/execution_time:.1f} cases/second")
print(f"Results include complete tritium inventory dynamics and economic analysis.")

# Data Visualization and Analysis

Now let's create comprehensive visualizations to explore the parametric analysis results and understand the relationships between parameters and economic outcomes.

In [ ]:
# COMPREHENSIVE PLOTTING FUNCTIONS FOR PARAMETRIC ANALYSIS
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("🎨 PARAMETRIC ANALYSIS VISUALIZATION TOOLKIT")
print("=" * 60)

# Check if we have data to plot
if 'comprehensive_df' in locals() and len(comprehensive_df) > 0:
    df = comprehensive_df.copy()
    print(f"✅ Using comprehensive analysis data: {len(df)} cases")
elif 'parallel_results_df' in locals() and len(parallel_results_df) > 0:
    df = parallel_results_df.copy()
    print(f"✅ Using parallel analysis data: {len(df)} cases")
else:
    print("❌ No analysis data found. Please run the parametric analysis first.")
    df = None

if df is not None:
    # Filter successful cases
    successful_df = df[df['success'] == True].copy()
    print(f"📊 Successfully analyzed cases: {len(successful_df)}")
    
    # Identify available columns for analysis
    key_params = ['V_plasma', 'T_i', 'n_tot', 'tau_p_T', 'TBR_DT', 'TBR_DDn', 'eta_th', 'plant_avail']
    key_outputs = ['t_startup_hours', 'Dollar_Lost', 'E_lost_MJ', 'P_DT_full_MW', 'Q_DT_full']
    
    available_params = [col for col in key_params if col in successful_df.columns]
    available_outputs = [col for col in key_outputs if col in successful_df.columns]
    
    print(f"📈 Available parameters: {available_params}")
    print(f"📉 Available outputs: {available_outputs}")
else:
    print("⚠️  No data available for plotting.")

In [ ]:
# 1. CORRELATION ANALYSIS AND HEATMAPS
if df is not None and len(successful_df) > 0:
    
    print("\n🔍 CORRELATION ANALYSIS")
    print("-" * 40)
    
    # Select numeric columns for correlation
    numeric_cols = available_params + available_outputs
    correlation_data = successful_df[numeric_cols].select_dtypes(include=[np.number])
    
    # Calculate correlation matrix
    correlation_matrix = correlation_data.corr()
    
    # Create correlation heatmap
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, 
                mask=mask,
                annot=True, 
                cmap='RdBu_r', 
                center=0,
                square=True,
                fmt='.2f',
                cbar_kws={"shrink": .8})
    plt.title('Parameter-Output Correlation Matrix', fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Print strongest correlations with economic outcomes
    if 'Dollar_Lost' in correlation_data.columns:
        dollar_correlations = correlation_matrix['Dollar_Lost'].abs().sort_values(ascending=False)
        print(f"\n💰 Strongest correlations with Dollar Lost:")
        for param, corr in dollar_correlations.head(6).items():
            if param != 'Dollar_Lost':
                direction = "increases" if correlation_matrix.loc['Dollar_Lost', param] > 0 else "decreases"
                print(f"  {param}: {corr:.3f} (Dollar Lost {direction} with {param})")
    
    if 't_startup_hours' in correlation_data.columns:
        startup_correlations = correlation_matrix['t_startup_hours'].abs().sort_values(ascending=False)
        print(f"\n⏰ Strongest correlations with Startup Time:")
        for param, corr in startup_correlations.head(6).items():
            if param != 't_startup_hours':
                direction = "increases" if correlation_matrix.loc['t_startup_hours', param] > 0 else "decreases"
                print(f"  {param}: {corr:.3f} (Startup time {direction} with {param})")

In [ ]:
# 2. SCATTER PLOTS AND PARAMETER SENSITIVITY
if df is not None and len(successful_df) > 0:
    
    print("\n\n📊 PARAMETER SENSITIVITY ANALYSIS")
    print("-" * 40)
    
    # Create subplot grid for key parameter relationships
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Parameter Sensitivity Analysis: Key Relationships', fontsize=16)
    
    # Define key relationships to plot
    relationships = []
    if 'Dollar_Lost' in successful_df.columns:
        if 'T_i' in successful_df.columns:
            relationships.append(('T_i', 'Dollar_Lost', 'Ion Temperature vs Dollar Lost'))
        if 'TBR_DT' in successful_df.columns:
            relationships.append(('TBR_DT', 'Dollar_Lost', 'TBR_DT vs Dollar Lost'))
        if 'TBR_DDn' in successful_df.columns:
            relationships.append(('TBR_DDn', 'Dollar_Lost', 'TBR_DDn vs Dollar Lost'))
    
    if 't_startup_hours' in successful_df.columns:
        if 'n_tot' in successful_df.columns:
            relationships.append(('n_tot', 't_startup_hours', 'Density vs Startup Time'))
        if 'eta_th' in successful_df.columns:
            relationships.append(('eta_th', 't_startup_hours', 'Thermal Efficiency vs Startup Time'))
        if 'plant_avail' in successful_df.columns:
            relationships.append(('plant_avail', 't_startup_hours', 'Plant Availability vs Startup Time'))
    
    # Plot relationships
    for i, (x_col, y_col, title) in enumerate(relationships[:6]):
        row, col = i // 3, i % 3
        ax = axes[row, col]
        
        # Create scatter plot with color mapping
        if len(available_params) > 2:
            color_col = available_params[2] if available_params[2] not in [x_col, y_col] else available_params[0]
            scatter = ax.scatter(successful_df[x_col], successful_df[y_col], 
                               c=successful_df[color_col], cmap='viridis', alpha=0.7, s=50)
            plt.colorbar(scatter, ax=ax, label=color_col)
        else:
            ax.scatter(successful_df[x_col], successful_df[y_col], alpha=0.7, s=50)
        
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title(title, fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for i in range(len(relationships), 6):
        row, col = i // 3, i % 3
        axes[row, col].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Interactive Plotly scatter plot for Dollar Lost vs key parameters
    if 'Dollar_Lost' in successful_df.columns and len(available_params) >= 2:
        print("\n💡 Creating interactive scatter plot...")
        
        fig_plotly = px.scatter(
            successful_df, 
            x=available_params[0], 
            y='Dollar_Lost',
            color=available_params[1] if len(available_params) > 1 else None,
            size='t_startup_hours' if 't_startup_hours' in successful_df.columns else None,
            hover_data=available_params[:4],
            title=f"Interactive: {available_params[0]} vs Dollar Lost",
            labels={'Dollar_Lost': 'Dollar Lost ($)'},
            template='plotly_white'
        )
        fig_plotly.update_layout(height=600)
        fig_plotly.show()

In [ ]:
# 3. DISTRIBUTION ANALYSIS AND BOX PLOTS
if df is not None and len(successful_df) > 0:
    
    print("\n\n📈 DISTRIBUTION ANALYSIS")
    print("-" * 40)
    
    # Distribution plots for key outputs
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Output Distributions and Statistical Analysis', fontsize=16)
    
    # Plot distributions
    plot_cols = [col for col in ['t_startup_hours', 'Dollar_Lost', 'E_lost_MJ', 'Q_DT_full'] 
                if col in successful_df.columns]
    
    for i, col in enumerate(plot_cols[:4]):
        row, col_idx = i // 2, i % 2
        ax = axes[row, col_idx]
        
        # Histogram with KDE
        successful_df[col].hist(bins=30, alpha=0.7, ax=ax, density=True)
        successful_df[col].plot.density(ax=ax, color='red', linewidth=2)
        
        ax.set_title(f'Distribution of {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Density')
        ax.grid(True, alpha=0.3)
        
        # Add statistics text
        mean_val = successful_df[col].mean()
        median_val = successful_df[col].median()
        std_val = successful_df[col].std()
        
        stats_text = f'Mean: {mean_val:.2e}\\nMedian: {median_val:.2e}\\nStd: {std_val:.2e}'
        ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, 
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Hide empty subplots
    for i in range(len(plot_cols), 4):
        row, col_idx = i // 2, i % 2
        axes[row, col_idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Box plots for parameter ranges vs economic outcomes
    if 'Dollar_Lost' in successful_df.columns and len(available_params) >= 2:
        print("\n📦 Parameter Range Analysis...")
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle('Parameter Ranges vs Economic Outcomes', fontsize=16)
        
        # Create categorical bins for continuous variables
        for i, param in enumerate(available_params[:3]):
            ax = axes[i]
            
            # Create bins for the parameter
            param_data = successful_df[param]
            bins = pd.qcut(param_data, q=3, labels=['Low', 'Medium', 'High'], duplicates='drop')
            
            # Box plot
            box_data = [successful_df[successful_df[param].between(param_data.quantile(j/3), 
                                                                  param_data.quantile((j+1)/3))]
                       ['Dollar_Lost'].values for j in range(3)]
            
            ax.boxplot([data for data in box_data if len(data) > 0], 
                      labels=['Low', 'Medium', 'High'])
            ax.set_title(f'{param} Ranges vs Dollar Lost')
            ax.set_xlabel(f'{param} Level')
            ax.set_ylabel('Dollar Lost ($)')
            ax.grid(True, alpha=0.3)
            
            # Add sample sizes
            for j, data in enumerate(box_data):
                if len(data) > 0:
                    ax.text(j+1, ax.get_ylim()[1], f'n={len(data)}', 
                           ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        plt.show()

In [ ]:
# 4. 3D VISUALIZATION AND OPTIMIZATION ANALYSIS
if df is not None and len(successful_df) > 0:
    
    print("\n\n🎯 OPTIMIZATION ANALYSIS")
    print("-" * 40)
    
    # 3D scatter plot for three key parameters
    if len(available_params) >= 3 and 'Dollar_Lost' in successful_df.columns:
        
        fig_3d = go.Figure(data=[go.Scatter3d(
            x=successful_df[available_params[0]],
            y=successful_df[available_params[1]],
            z=successful_df[available_params[2]],
            mode='markers',
            marker=dict(
                size=5,
                color=successful_df['Dollar_Lost'],
                colorscale='Viridis',
                colorbar=dict(title="Dollar Lost ($)"),
                showscale=True
            ),
            text=[f'{available_params[0]}: {x:.2f}<br>{available_params[1]}: {y:.2f}<br>{available_params[2]}: {z:.2f}<br>Dollar Lost: ${w:.0f}'
                  for x, y, z, w in zip(successful_df[available_params[0]], 
                                      successful_df[available_params[1]], 
                                      successful_df[available_params[2]], 
                                      successful_df['Dollar_Lost'])],
            hovertemplate='%{text}<extra></extra>'
        )])
        
        fig_3d.update_layout(
            title=f'3D Parameter Space: {available_params[0]} vs {available_params[1]} vs {available_params[2]}',
            scene=dict(
                xaxis_title=available_params[0],
                yaxis_title=available_params[1],
                zaxis_title=available_params[2]
            ),
            height=700
        )
        fig_3d.show()
    
    # Pareto frontier analysis for multi-objective optimization
    if 'Dollar_Lost' in successful_df.columns and 't_startup_hours' in successful_df.columns:
        print("\n🎯 Pareto Frontier Analysis (Dollar Lost vs Startup Time)")
        
        # Calculate Pareto frontier
        def is_pareto_efficient(costs):
            """Find Pareto efficient points"""
            is_efficient = np.arange(costs.shape[0])
            n_points = costs.shape[0]
            next_point_index = 0
            
            while next_point_index < len(costs):
                nondominated_point_mask = np.any(costs < costs[next_point_index], axis=1)
                nondominated_point_mask[next_point_index] = True
                is_efficient = is_efficient[nondominated_point_mask]
                costs = costs[nondominated_point_mask]
                next_point_index = np.sum(nondominated_point_mask[:next_point_index]) + 1
            
            return is_efficient
        
        # Prepare data for Pareto analysis (minimize both objectives)
        objectives = successful_df[['Dollar_Lost', 't_startup_hours']].values
        pareto_indices = is_pareto_efficient(objectives)
        
        # Plot Pareto frontier
        plt.figure(figsize=(12, 8))
        
        # All points
        plt.scatter(successful_df['t_startup_hours'], successful_df['Dollar_Lost'], 
                   alpha=0.6, s=50, c='lightblue', label='All Solutions')
        
        # Pareto efficient points
        pareto_df = successful_df.iloc[pareto_indices]
        plt.scatter(pareto_df['t_startup_hours'], pareto_df['Dollar_Lost'], 
                   alpha=0.8, s=100, c='red', marker='*', label='Pareto Efficient', zorder=5)
        
        # Connect Pareto points
        pareto_sorted = pareto_df.sort_values('t_startup_hours')
        plt.plot(pareto_sorted['t_startup_hours'], pareto_sorted['Dollar_Lost'], 
                'r--', alpha=0.7, linewidth=2, label='Pareto Frontier')
        
        plt.xlabel('Startup Time (hours)')
        plt.ylabel('Dollar Lost ($)')
        plt.title('Multi-Objective Optimization: Pareto Frontier Analysis')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
        plt.tight_layout()
        plt.show()
        
        print(f"📊 Found {len(pareto_indices)} Pareto efficient solutions out of {len(successful_df)} total solutions")
        print(f"📈 Pareto efficiency rate: {len(pareto_indices)/len(successful_df)*100:.1f}%")
        
        # Show best Pareto solutions
        print("\\n🏆 Top 5 Pareto Efficient Solutions:")
        pareto_summary = pareto_df[['t_startup_hours', 'Dollar_Lost'] + available_params[:3]].round(3)
        print(pareto_summary.head())

In [ ]:
# 5. PARAMETER IMPORTANCE AND SUMMARY ANALYSIS
if df is not None and len(successful_df) > 0:
    
    print("\n\n📊 PARAMETER IMPORTANCE ANALYSIS")
    print("-" * 40)
    
    # Calculate parameter importance using correlation and variance
    if 'Dollar_Lost' in successful_df.columns:
        
        # Parameter importance based on correlation with Dollar Lost
        correlations = correlation_matrix['Dollar_Lost'].abs().drop('Dollar_Lost').sort_values(ascending=False)
        
        # Parameter importance based on variance explanation
        variances = {}
        for param in available_params:
            if param in successful_df.columns:
                param_range = successful_df[param].max() - successful_df[param].min()
                param_std = successful_df[param].std()
                variances[param] = param_std / param_range if param_range > 0 else 0
        
        # Create importance plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Correlation importance
        correlations.plot(kind='barh', ax=ax1, color='skyblue')
        ax1.set_title('Parameter Importance: Correlation with Dollar Lost')
        ax1.set_xlabel('Absolute Correlation')
        ax1.grid(True, alpha=0.3)
        
        # Variance importance
        var_series = pd.Series(variances).sort_values(ascending=False)
        var_series.plot(kind='barh', ax=ax2, color='lightcoral')
        ax2.set_title('Parameter Importance: Normalized Variance')
        ax2.set_xlabel('Normalized Standard Deviation')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("🎯 Parameter Ranking by Impact on Dollar Lost:")
        for i, (param, corr) in enumerate(correlations.head().items(), 1):
            print(f"  {i}. {param}: {corr:.3f} correlation")
    
    # Summary statistics table
    print("\\n\\n📋 SUMMARY STATISTICS")
    print("-" * 40)
    
    summary_cols = [col for col in ['t_startup_hours', 'Dollar_Lost', 'E_lost_MJ'] + available_params 
                   if col in successful_df.columns]
    
    summary_stats = successful_df[summary_cols].describe()
    print("Statistical Summary:")
    print(summary_stats.round(3))
    
    # Performance metrics
    if 't_startup_hours' in successful_df.columns:
        fast_startup = (successful_df['t_startup_hours'] < 0.1).sum()
        print(f"\\n⚡ Fast startup cases (< 0.1 hours): {fast_startup} ({fast_startup/len(successful_df)*100:.1f}%)")
    
    if 'Dollar_Lost' in successful_df.columns:
        low_cost = (successful_df['Dollar_Lost'] < successful_df['Dollar_Lost'].median()).sum()
        print(f"💰 Low-cost cases (below median): {low_cost} ({low_cost/len(successful_df)*100:.1f}%)")
    
    print(f"\\n🎉 Analysis complete! Generated comprehensive visualizations for {len(successful_df)} successful cases.")

# Create a function for custom plotting
def plot_custom_relationship(x_param, y_param, color_param=None, size_param=None):
    """
    Create a custom scatter plot for any parameter relationship.
    
    Parameters:
    - x_param: Column name for x-axis
    - y_param: Column name for y-axis  
    - color_param: Column name for color mapping (optional)
    - size_param: Column name for size mapping (optional)
    """
    if df is None or successful_df is None:
        print("❌ No data available. Run the parametric analysis first.")
        return
    
    available_cols = successful_df.columns.tolist()
    
    if x_param not in available_cols or y_param not in available_cols:
        print(f"❌ Parameters not found. Available columns: {available_cols}")
        return
    
    plt.figure(figsize=(10, 7))
    
    if color_param and color_param in available_cols:
        scatter = plt.scatter(successful_df[x_param], successful_df[y_param], 
                            c=successful_df[color_param], cmap='viridis', 
                            s=successful_df[size_param]*50 if size_param and size_param in available_cols else 50,
                            alpha=0.7)
        plt.colorbar(scatter, label=color_param)
    else:
        plt.scatter(successful_df[x_param], successful_df[y_param], alpha=0.7)
    
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.title(f'{x_param} vs {y_param}' + (f' (colored by {color_param})' if color_param else ''))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\\n🛠️  Custom plotting function available:")
print("   plot_custom_relationship('parameter1', 'parameter2', 'color_param', 'size_param')")
print("\\nExample usage:")
print("   plot_custom_relationship('T_i', 'Dollar_Lost', 'TBR_DT', 't_startup_hours')")

# 🎨 Comprehensive Visualization Summary

## Plotting Capabilities Created

You now have a complete visualization toolkit for your parametric analysis results! Here's what's available:

### 📊 **1. Correlation Analysis & Heatmaps**
- **Purpose**: Identify which parameters most strongly affect economic outcomes
- **Key Insights**: Shows that startup time is most negatively correlated with TBR_DT (-0.56), and Dollar Lost increases strongly with energy lost (0.88)

### 📈 **2. Parameter Sensitivity Analysis**
- **Purpose**: Scatter plots showing relationships between key parameters and outputs
- **Features**: Interactive Plotly plots with hover data and color/size mapping
- **Usage**: Identify optimal parameter ranges and trade-offs

### 📦 **3. Distribution Analysis & Box Plots**
- **Purpose**: Understand output distributions and parameter range effects
- **Features**: Histograms with KDE, statistical summaries, categorical analysis
- **Insights**: Shows whether parameters have normal/skewed distributions

### 🎯 **4. 3D Visualization & Optimization**
- **Purpose**: Multi-dimensional parameter space exploration
- **Features**: Interactive 3D scatter plots, Pareto frontier analysis
- **Applications**: Multi-objective optimization, trade-off analysis

### 📊 **5. Parameter Importance Analysis**
- **Purpose**: Rank parameters by their impact on outcomes
- **Features**: Correlation-based and variance-based importance metrics
- **Results**: Comprehensive summary statistics and performance metrics

### 🛠️ **Custom Plotting Function**
```python
plot_custom_relationship('T_i', 'Dollar_Lost', 'TBR_DT', 't_startup_hours')
```

## 🔍 Key Findings from Your Analysis

**Most Important Parameters for Economic Optimization:**
1. **TBR_DT**: Higher breeding ratios significantly reduce startup time
2. **Energy Lost (E_lost_MJ)**: Directly drives economic losses  
3. **Ion Temperature (T_i)**: Higher temperatures increase dollar lost
4. **Plasma Volume & Density**: Strong correlation with power output

**Optimization Insights:**
- **Best Strategy**: Maximize TBR_DT and TBR_DDn, minimize ion temperature
- **Trade-offs**: Higher power density increases losses but improves performance
- **Economic Impact**: Energy losses during startup are the primary cost driver

## 📋 Usage Instructions

1. **Run all visualization cells** to generate comprehensive plots
2. **Use the custom function** for specific parameter relationships
3. **Examine Pareto frontier** for multi-objective optimization
4. **Check correlation heatmap** to identify unexpected relationships
5. **Use 3D plots** for complex parameter interactions

The analysis provides both **statistical insights** and **visual understanding** of your tritium inventory system's behavior across the full parameter space!

In [ ]:
# 🌐 PARALLEL COORDINATES VISUALIZATION - ALL INPUT PARAMETERS + DOLLAR LOST
# ==============================================================================

def create_parallel_coordinates_all_params(df, color_by='Dollar_Lost'):
    """
    Create comprehensive parallel coordinates plot with ALL input parameters + Dollar Lost
    """
    print("\n🌐 PARALLEL COORDINATES VISUALIZATION")
    print("-" * 50)
    
    # Define all input parameter columns to include + Dollar Lost as rightmost axis
    input_params = [
        'V_plasma', 'T_i', 'n_tot', 'tau_p_T', 'tau_p_He3', 
        'P_aux', 'P_lost_rad', 'P_aux_all_DT', 'P_lost_rad_all_DT',
        'TBR_DT', 'TBR_DDn', 'tau_ifc', 'tau_ofc', 'eta_th',
        'plant_avail', 'Cost_per_kWh',  # Using actual column names
        'Dollar_Lost'  # Add Dollar Lost as rightmost axis
    ]
    
    # Select only the parameters that exist in our data
    available_params = [param for param in input_params if param in df.columns]
    print(f"📊 Creating parallel coordinates with {len(available_params)} parameters (including Dollar Lost)")
    
    # Set up color coding
    if color_by == 'Dollar_Lost':
        color_values = df['Dollar_Lost'] / 1e9  # Convert to billions
        color_label = 'Dollar Lost (B$)'
    else:
        color_values = df[color_by]
        color_label = color_by
    
    print(f"🎯 Color-coded by: {color_label}")
    
    # Create dimensions for each parameter
    dimensions = []
    
    # Define parameter labels and units
    param_info = {
        'V_plasma': {'label': 'V_plasma [m³]', 'unit': 'm³'},
        'T_i': {'label': 'T_i [keV]', 'unit': 'keV'},
        'n_tot': {'label': 'n_tot [×10²⁰ m⁻³]', 'unit': '×10²⁰ m⁻³'},
        'tau_p_T': {'label': 'τ_p_T [s]', 'unit': 's'},
        'tau_p_He3': {'label': 'τ_p_He3 [s]', 'unit': 's'},
        'P_aux': {'label': 'P_aux [MW]', 'unit': 'MW'},
        'P_lost_rad': {'label': 'P_lost_rad [MW]', 'unit': 'MW'},
        'P_aux_all_DT': {'label': 'P_aux_all_DT [MW]', 'unit': 'MW'},
        'P_lost_rad_all_DT': {'label': 'P_lost_rad_all_DT [MW]', 'unit': 'MW'},
        'TBR_DT': {'label': 'TBR_DT', 'unit': ''},
        'TBR_DDn': {'label': 'TBR_DDn', 'unit': ''},
        'tau_ifc': {'label': 'τ_ifc [h]', 'unit': 'h'},
        'tau_ofc': {'label': 'τ_ofc [h]', 'unit': 'h'},
        'eta_th': {'label': 'η_th', 'unit': ''},
        'plant_avail': {'label': 'Plant Availability', 'unit': ''},
        'Cost_per_kWh': {'label': 'Cost [USD/kWh]', 'unit': 'USD/kWh'},
        'Dollar_Lost': {'label': 'Dollar Lost [$]', 'unit': '$'}  # Add Dollar Lost info
    }
    
    for param in available_params:
        values = df[param]
        unique_vals = sorted(values.unique())
        
        # Print parameter info
        param_label = param_info.get(param, {}).get('label', param)
        
        # Special formatting for Dollar_Lost
        if param == 'Dollar_Lost':
            print(f"  • {param_label}: {len(unique_vals)} unique values, range [${min(values):,.0f}, ${max(values):,.0f}]")
        else:
            print(f"  • {param_label}: {len(unique_vals)} unique values, range [{min(values):.3g}, {max(values):.3g}]")
        
        # Create dimension
        if param == 'Dollar_Lost':
            # Special handling for Dollar Lost - show fewer tick marks for readability
            dimension = dict(
                range=[min(values), max(values)],
                label=param_label,
                values=values,
                tickvals=None,  # Let plotly auto-select for large ranges
                ticktext=None
            )
        else:
            dimension = dict(
                range=[min(values), max(values)],
                label=param_label,
                values=values,
                tickvals=unique_vals if len(unique_vals) <= 10 else None,
                ticktext=[f"{val:.3g}" for val in unique_vals] if len(unique_vals) <= 10 else None
            )
        dimensions.append(dimension)
    
    # Set colorscale based on color values
    if color_by == 'Dollar_Lost':
        colorscale = 'Reds'  # Red scale for losses
    else:
        colorscale = 'Viridis'
    
    # Create the parallel coordinates plot
    fig = go.Figure(
        data=go.Parcoords(
            line=dict(
                color=color_values,
                colorscale=colorscale,
                showscale=True,
                colorbar=dict(
                    title=color_label,
                    tickmode="linear",
                    thickness=20,
                    len=0.8
                )
            ),
            dimensions=dimensions,
            labelangle=45,
            labelside="top"
        )
    )
    
    # Update layout
    fig.update_layout(
        title={
            'text': f"Parallel Coordinates: All Input Parameters + {color_label}",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16}
        },
        width=1500,  # Increased width to accommodate Dollar Lost axis
        height=600,
        font=dict(size=12)
    )
    
    return fig

# Check what data we have
print("📊 Available dataframes:")
print(f"  • comprehensive_df shape: {comprehensive_df.shape}")
print(f"  • Key columns: {[col for col in comprehensive_df.columns if any(keyword in col.lower() for keyword in ['dollar', 'cost', 'lost'])]}")

# Create and display the comprehensive parallel coordinates plot
fig_parallel_all = create_parallel_coordinates_all_params(
    comprehensive_df, 
    color_by='Dollar_Lost'
)

fig_parallel_all.show()

print("\n✅ Comprehensive parallel coordinates plot created!")
print("🔍 This plot shows relationships between ALL input parameters + Dollar Lost")
print("📈 Color coding represents the economic impact (Dollar Lost)")
print("🎯 Each line represents one parameter combination")
print("📊 Dollar Lost is now displayed as the rightmost axis")
print("💰 You can see both the input parameters and the resulting economic impact")